# Synthetic data generator — Zero Traffic Alert

Synthetic data generator for the Zero Traffic Alert portfolio project.

Produces a daily casino x provider traffic table that reproduces the *statistical
shape* of a real B2B gaming operations problem while containing no real data:
every casino, provider, account manager, date and figure is generated.

The dashboard this feeds is only interesting if the data contains the situations
it was designed to separate, so the generator builds them explicitly rather than
hoping they emerge from noise. Each casino is assigned one scenario:

    healthy          traffic on essentially every line, every day
    provider_gap     casino runs fine, but 1-3 of its providers have stopped
    intermittent     several providers drop in and out across the window
    casino_silent    the whole casino goes dark for 1-2 days in the final week
    casino_dead      no traffic at all across the whole window
    underperforming  technically alive, but negligible volume

Two further properties are built in deliberately, because both drove design
decisions in the dashboard:

    * Revenue is heavy-tailed (log-normal), spanning ~5 orders of magnitude.
      This is why the alert threshold has no natural break point.
    * One provider is enabled across most of the client base on a single date
      and never carries a transaction. This reproduces the batch-provisioning
      noise that dominated the original report.

Usage:
    python generate_synthetic_data.py                     # write CSV + validate
    python generate_synthetic_data.py --seed 7 --no-validate

## Setup and constants

In [1]:
from __future__ import annotations
import argparse
import datetime as dt
from dataclasses import dataclass, field
import numpy as np
import pandas as pd
WINDOW_START = dt.date(2026, 3, 1)
WINDOW_DAYS = 50
PROVIDERS = [
    "Provider 1 (Live)",   # "Provider 1" is split into two products to preserve
    "Provider 1 (RNG)",    # the first-party-vs-third-party structure that the
    "Provider 2",          # priority-provider logic depends on.
    "Provider 3",
    "Provider 4",
    "Provider 5",
    "Provider 6",
]
BATCH_PROVIDER = "Provider 7"       # enabled in bulk, never goes live
BATCH_ADOPTION = 0.62               # share of casinos it is enabled on
BATCH_GOLIVE_OFFSET = 6             # all on the same date -> a detectable batch
GROUP_LETTERS = list("ABCDEFGHIJKLMNOPQRST")
N_ACCOUNT_MANAGERS = 14
ENVIRONMENTS = ["ENV-A", "ENV-B"]
SCENARIO_MIX = {
    "healthy": 62,
    "provider_gap": 28,
    "casino_silent": 14,
    "casino_dead": 10,
    "intermittent": 10,
    "underperforming": 6,
}
BASELINE_DROPOUT = 0.0025   # per line per day: real lines rarely blink out
INTERMITTENT_DROPOUT = 0.40
UNDERPERFORMING_DROPOUT = 0.80
HARD_STOP_MAX_AGE = 20      # a hard stop began 1-20 days before the report day
WAGER_LOG_MEAN, WAGER_LOG_SIGMA = 7.4, 2.5
WAGER_DAILY_NOISE = 0.45
MARGIN_RANGE = (0.022, 0.065)   # GGR as a share of wager
MARGIN_NOISE = 0.25
COLUMNS = [
    "Environment code", "Casino ID", "Casino", "Casino group", "Single KAM",
    "Game Provider", "DateOnly", "Traffic Flag", "Grouping Set Level",
    "Wager Eur", "GGR Eur", "Go Live Date", "Casino Provider Go Live Date",
]


### `Casino`

In [2]:
@dataclass
class Casino:
    env: str
    casino_id: str
    name: str
    group: str
    kam: str
    scenario: str
    scale: float                       # baseline daily wager for the whole casino
    go_live: dt.date
    providers: list[str] = field(default_factory=list)
    shares: np.ndarray | None = None   # how the casino's volume splits by provider
    has_batch_provider: bool = False


### `build_casinos`

In [3]:
def build_casinos(rng: np.random.Generator) -> list[Casino]:
    """Create the casino population with scenarios assigned deterministically."""
    scenarios = [s for s, n in SCENARIO_MIX.items() for _ in range(n)]
    casinos: list[Casino] = []

    for i, scenario in enumerate(scenarios):
        letter = GROUP_LETTERS[i % len(GROUP_LETTERS)]
        sequence = i // len(GROUP_LETTERS) + 1

        # Underperforming casinos need to land under the cleanup threshold, so
        # their scale is drawn from a separate, tiny range rather than the tail.
        scale = (
            float(rng.uniform(0.5, 9.0)) if scenario == "underperforming"
            else float(np.exp(rng.normal(WAGER_LOG_MEAN, WAGER_LOG_SIGMA)))
        )

        n_providers = int(rng.integers(3, 7))
        providers = list(rng.choice(PROVIDERS, size=n_providers, replace=False))

        casinos.append(Casino(
            env=ENVIRONMENTS[i % len(ENVIRONMENTS)],
            casino_id=f"C{1000 + i}",
            name=f"Casino {letter}{sequence}",
            group=f"Group {letter}",
            kam=f"KAM {i % N_ACCOUNT_MANAGERS + 1:02d}",
            scenario=scenario,
            scale=scale,
            go_live=WINDOW_START - dt.timedelta(days=int(rng.integers(30, 900))),
            providers=providers,
            # Dirichlet with alpha < 1 concentrates volume on one or two lines,
            # which is what makes "the biggest broken line" a meaningful ranking.
            shares=rng.dirichlet(np.ones(n_providers) * 0.6),
            has_batch_provider=bool(rng.random() < BATCH_ADOPTION),
        ))

    return casinos


### `plan_outages`

In [4]:
def plan_outages(casino: Casino, rng: np.random.Generator) -> tuple[dict, dict, set]:
    """Decide, per casino, which lines break and when.

    Returns (break_mode_by_provider, hard_stop_day_by_provider, casino_dark_days).
    """
    break_mode: dict[str, str] = {}

    if casino.scenario == "provider_gap":
        n_broken = int(rng.integers(1, min(3, len(casino.providers)) + 1))
        for p in rng.choice(casino.providers, size=n_broken, replace=False):
            break_mode[p] = "hard" if rng.random() < 0.6 else "intermittent"

    elif casino.scenario == "intermittent":
        n_broken = max(1, len(casino.providers) // 2)
        for p in rng.choice(casino.providers, size=n_broken, replace=False):
            break_mode[p] = "intermittent"

    hard_stop_day = {
        p: WINDOW_DAYS - int(rng.integers(1, HARD_STOP_MAX_AGE + 1))
        for p, mode in break_mode.items() if mode == "hard"
    }

    casino_dark_days: set[int] = set()
    if casino.scenario == "casino_silent":
        casino_dark_days = set(rng.choice(
            range(WINDOW_DAYS - 7, WINDOW_DAYS),
            size=int(rng.integers(1, 3)), replace=False,
        ))

    return break_mode, hard_stop_day, casino_dark_days


### `is_active`

In [5]:
def is_active(casino: Casino, provider: str, day_index: int, break_mode: dict,
              hard_stop_day: dict, casino_dark_days: set,
              rng: np.random.Generator) -> bool:
    """Did this line carry traffic on this day?  Order matters: casino-wide
    conditions override provider-level ones, exactly as they do in reality."""
    if casino.scenario == "casino_dead":
        return False
    if casino.scenario == "underperforming" and rng.random() < UNDERPERFORMING_DROPOUT:
        return False
    if day_index in casino_dark_days:
        return False
    if provider in hard_stop_day and day_index >= hard_stop_day[provider]:
        return False
    if break_mode.get(provider) == "intermittent" and rng.random() < INTERMITTENT_DROPOUT:
        return False
    return rng.random() >= BASELINE_DROPOUT


### `generate`

In [6]:
def generate(seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    days = [WINDOW_START + dt.timedelta(days=i) for i in range(WINDOW_DAYS)]
    batch_go_live = WINDOW_START + dt.timedelta(days=BATCH_GOLIVE_OFFSET)

    casinos = build_casinos(rng)
    rows: list[list] = []

    for casino in casinos:
        break_mode, hard_stop_day, casino_dark_days = plan_outages(casino, rng)
        casino_totals: dict[dt.date, list[float]] = {}

        # --- casino x provider grain -------------------------------------- #
        for provider, share in zip(casino.providers, casino.shares):
            provider_go_live = casino.go_live + dt.timedelta(days=int(rng.integers(0, 60)))

            for day_index, day in enumerate(days):
                active = is_active(casino, provider, day_index, break_mode,
                                   hard_stop_day, casino_dark_days, rng)

                if not active:
                    # No-traffic rows carry null money, matching how the source
                    # system's outer join manufactures them.
                    rows.append([casino.env, casino.casino_id, casino.name,
                                 casino.group, casino.kam, provider, day,
                                 "No Traffic", "casino-provider", None, None,
                                 casino.go_live, provider_go_live])
                    continue

                wager = casino.scale * float(share) * float(np.exp(
                    rng.normal(0, WAGER_DAILY_NOISE)))
                ggr = wager * float(rng.uniform(*MARGIN_RANGE)) * float(np.exp(
                    rng.normal(0, MARGIN_NOISE)))

                rows.append([casino.env, casino.casino_id, casino.name,
                             casino.group, casino.kam, provider, day,
                             "Have Traffic", "casino-provider",
                             round(wager, 2), round(ggr, 2),
                             casino.go_live, provider_go_live])

                totals = casino_totals.setdefault(day, [0.0, 0.0])
                totals[0] += wager
                totals[1] += ggr

        # --- the batch-provisioned line that never goes live --------------- #
        if casino.has_batch_provider:
            for day in days:
                rows.append([casino.env, casino.casino_id, casino.name,
                             casino.group, casino.kam, BATCH_PROVIDER, day,
                             "No Traffic", "casino-provider", None, None,
                             casino.go_live, batch_go_live])

        # --- casino grain (the source's second grouping set) --------------- #
        for day in days:
            totals = casino_totals.get(day)
            rows.append([casino.env, casino.casino_id, casino.name,
                         casino.group, casino.kam, None, day,
                         "Have Traffic" if totals else "No Traffic", "casino",
                         round(totals[0], 2) if totals else None,
                         round(totals[1], 2) if totals else None,
                         casino.go_live, None])

    return pd.DataFrame(rows, columns=COLUMNS)


### `validate`

In [7]:
def validate(df: pd.DataFrame, cleanup_threshold: float = 500.0) -> None:
    """Re-implement the dashboard's routing logic in pandas and report the
    resulting population. If these four tables do not overlap and none of them
    is empty, the dataset exercises every branch of the dashboard."""
    cp = df[df["Grouping Set Level"] == "casino-provider"].copy()
    cp["DateOnly"] = pd.to_datetime(cp["DateOnly"])
    report_day = cp["DateOnly"].max()
    start_7 = report_day - pd.Timedelta(days=6)
    start_50 = report_day - pd.Timedelta(days=49)

    cp["has_traffic"] = cp["Traffic Flag"].eq("Have Traffic")
    cp["traffic_date"] = np.where(cp["has_traffic"], cp["DateOnly"], pd.NaT)

    ids = sorted(cp["Casino ID"].unique())
    last_traffic = cp.groupby("Casino ID")["traffic_date"].max().reindex(ids)
    wager_50d = (cp[cp["DateOnly"] >= start_50].groupby("Casino ID")["Wager Eur"]
                 .sum().reindex(ids).fillna(0))
    casino_zero_days = 7 - (cp[cp["DateOnly"] >= start_7]
                            .groupby(["Casino ID", "DateOnly"])["has_traffic"].any()
                            .groupby("Casino ID").sum().reindex(ids).fillna(0))
    provider_zero_days = 7 - (cp[cp["DateOnly"] >= start_7]
                              .groupby(["Casino ID", "Game Provider"])["has_traffic"].sum())
    provider_days_50d = (cp[cp["DateOnly"] >= start_50]
                         .groupby(["Casino ID", "Game Provider"])["has_traffic"].sum())

    # Only providers that have actually carried traffic can put a casino on the
    # list -- see "Known issue" in docs/calculated-fields.md.
    stopped_provider = ((provider_zero_days[provider_days_50d > 0] >= 1)
                        .groupby("Casino ID").max().reindex(ids).fillna(False))

    never_live = last_traffic.isna()
    underperforming = ~never_live & (wager_50d < cleanup_threshold)
    remaining = ~never_live & ~underperforming
    casino_dark = remaining & (casino_zero_days >= 1)
    provider_gap = remaining & (casino_zero_days == 0) & stopped_provider
    listed = never_live | underperforming | casino_dark | provider_gap

    n = len(ids)
    print(f"\n{len(df):,} rows · {n} casinos · "
          f"{cp['Game Provider'].nunique()} providers · {WINDOW_DAYS} days")
    print(f"Window: {report_day.date() - dt.timedelta(days=WINDOW_DAYS - 1)} "
          f"to {report_day.date()}\n")
    print(f"{'Routing':<38}{'casinos':>9}{'share':>9}")
    print("-" * 56)
    for label, mask in [
        ("(1) Never live in the window", never_live),
        ("(2) Underperforming", underperforming),
        ("(3) Whole casino dark this week", casino_dark),
        ("(4) Provider stopped, casino healthy", provider_gap),
        ("    Healthy - not listed", ~listed),
    ]:
        print(f"{label:<38}{int(mask.sum()):>9}{mask.sum() / n:>9.1%}")

    overlaps = int((never_live.astype(int) + underperforming
                    + casino_dark + provider_gap > 1).sum())
    n_kams = df["Single KAM"].nunique()
    actionable = int((casino_dark | provider_gap).sum())
    print("-" * 56)
    print(f"{'Casinos routed to two tables':<38}{overlaps:>9}"
          f"   <- must be 0")
    print(f"{'Needing action this week (3+4)':<38}{actionable:>9}"
          f"{actionable / n_kams:>8.1f} per KAM")

    never_used = (provider_days_50d == 0).groupby("Game Provider").sum()
    batch_share = never_used.get(BATCH_PROVIDER, 0) / max(never_used.sum(), 1)
    print(f"\nNever-used lines: {int(never_used.sum())} "
          f"({batch_share:.0%} from {BATCH_PROVIDER} alone)")

    avg_daily = (cp.groupby(["Casino ID", "Game Provider"])["Wager Eur"].sum()
                 / provider_days_50d.replace(0, np.nan))
    print(f"Average daily wager per line: {avg_daily.min():,.0f} "
          f"to {avg_daily.max():,.0f}  (heavy-tailed, no natural cut point)")


## Run

Writes `traffic_monitor_synthetic.csv` next to this notebook and prints the validation summary. Change the seed for a different but equally shaped dataset.

In [8]:
def main(argv=None) -> None:
    parser = argparse.ArgumentParser(description="Synthetic data generator for the Zero Traffic Alert portfolio project.")
    parser.add_argument("--seed", type=int, default=20260921)
    parser.add_argument("--out", default="traffic_monitor_synthetic.csv")
    parser.add_argument("--no-validate", action="store_true")
    args = parser.parse_args(argv)

    df = generate(args.seed)
    df.to_csv(args.out, index=False)
    print(f"Wrote {args.out}")

    if not args.no_validate:
        validate(df)


In [9]:
main(["--seed", "20260921"])   # the default seed — reproduces the CSV in the repo exactly

Wrote traffic_monitor_synthetic.csv

39,600 rows · 130 casinos · 8 providers · 50 days
Window: 2026-03-01 to 2026-04-19

Routing                                 casinos    share
--------------------------------------------------------
(1) Never live in the window                 10     7.7%
(2) Underperforming                          12     9.2%
(3) Whole casino dark this week              14    10.8%
(4) Provider stopped, casino healthy         39    30.0%
    Healthy - not listed                     55    42.3%
--------------------------------------------------------
Casinos routed to two tables                  0   <- must be 0
Needing action this week (3+4)               53     3.8 per KAM

Never-used lines: 121 (70% from Provider 7 alone)
Average daily wager per line: 0 to 330,387  (heavy-tailed, no natural cut point)
